# **01_EXT — Extracción e Ingestión de Datos (Capa Bronze)**

## Descripción
Esta notebook implementa la capa Bronze de la arquitectura Medallón.
Se realiza la extracción del dataset fuente desde el archivo CSV original 
y su persistencia en el Lakehouse como tabla Delta, aplicando únicamente 
la normalización de nombres de columnas requerida por las reglas de gobernanza.

## Criterios de la Capa Bronze
| Criterio | Descripción |
|---|---|
| Dato fuente intacto | No se aplican filtros, limpiezas ni transformaciones de negocio |
| Normalización de nombres | Columnas estandarizadas a `snake_case` |
| Formato Delta | Persistencia en formato Delta para trazabilidad y versionamiento |
| Idempotencia | Modo `overwrite` permite re-ejecución sin duplicados |

## Origen del Dato
- **Archivo:** `Files/retail_sales_dataset.csv`
- **Fuente:** [Kaggle — Retail Sales Dataset](https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset)
- **Tabla destino:** `Bronze.ventas_raw`

## Flujo de la Notebook
1. Carga del CSV original con encabezados
2. Normalización de nombres de columnas a `snake_case`
3. Persistencia como tabla Delta en Bronze


## **Extracción del Dataset Fuente**
Se carga el archivo CSV original desde la zona de archivos del Lakehouse 
sin aplicar ninguna transformación, preservando el dato fuente tal como 
fue recibido del cliente.

> 📂 Origen: `Files/retail_sales_dataset.csv`  
> 🔗 Fuente: [Kaggle — Retail Sales Dataset](https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset)

In [3]:
df= spark.read.csv("Files/retail_sales_dataset.csv" ,header=True)
display(df)

StatementMeta(, 0922c8fc-39da-4dc5-a301-120f965b68ac, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c32c7af4-7331-447b-8333-73a90a1617ad)

In [4]:
# Esto debería devolverte 1000
print(df.count())

StatementMeta(, 0922c8fc-39da-4dc5-a301-120f965b68ac, 6, Finished, Available, Finished, False)

1000


## **Normalización de Nombres y Persistencia en Bronze**
Se aplica la única transformación permitida en Bronze: estandarización 
de nombres de columnas a `snake_case` para cumplir las reglas de 
gobernanza del proyecto.

| Transformación | Ejemplo |
|---|---|
| Minúsculas | `Gender` → `gender` |
| Espacios a guión bajo | `Product Category` → `product_category` |
| Guiones a guión bajo | `Price-Per-Unit` → `price_per_unit` |

El resultado se persiste como tabla Delta `ventas_raw` en Bronze, 
conservando todos los registros originales sin filtros ni limpiezas.

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Convertir nombres de columnas a snake_case
for col in df.columns:
    nuevo_nombre = (
        col.strip()
           .lower()
           .replace(" ", "_")
           .replace("-", "_")
           .replace("/", "_")
    )
    df = df.withColumnRenamed(col, nuevo_nombre)

# Guardar en Bronze
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("ventas_raw")

display(df)

StatementMeta(, 0922c8fc-39da-4dc5-a301-120f965b68ac, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 395f52e1-18c4-4e47-9396-d22f7613afdd)

In [6]:
print(df)

StatementMeta(, 0922c8fc-39da-4dc5-a301-120f965b68ac, 8, Finished, Available, Finished, False)

DataFrame[transaction_id: string, date: string, customer_id: string, gender: string, age: string, product_category: string, quantity: string, price_per_unit: string, total_amount: string]


## **Confirmación de Persistencia en Bronze**
Escritura final de la tabla `ventas_raw` en formato Delta para garantizar 
que el dato fuente queda disponible para las capas posteriores.

> ⚠️ El modo `overwrite` garantiza idempotencia — el notebook puede 
> re-ejecutarse sin generar duplicados.

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


df.write.mode("overwrite")\
        .format("delta")\
        .saveAsTable("ventas_raw")

StatementMeta(, 0922c8fc-39da-4dc5-a301-120f965b68ac, 9, Finished, Available, Finished, False)